# FoundML 3 — Coding Assignment (Student Version)
## Autoencoder Representations and Identity Consistency (Olivetti Faces) — TensorFlow/Keras

You will train a **simple autoencoder** and evaluate how its learned latent representation behaves under
**competitive learning**, using the **Identity Consistency Score (ICS)** from the previous assignment.

This notebook is **moderate difficulty**:
- you implement/train a Keras autoencoder,
- you extract latent vectors,
- you run online k-means in latent space,
- you compute a small set of auto-gradable metrics.

All numerical answers must be rounded to **2 decimals**.


## Rules
- Use the provided random seeds and hyperparameters.
- Labels (`y`) are used **only for evaluation** (ICS), never during training.
- Do not change the grading variable names (Q1–Q4).


## 0. Setup

In [ ]:
# --- Determinism bootstrap: run this cell first after a kernel restart ---
import os, random
import numpy as np

SEED = 0

# Make Python hashing deterministic (best set before Python starts, but keep here as well)
os.environ["PYTHONHASHSEED"] = str(SEED)

# TensorFlow op-level determinism (must be set BEFORE importing tensorflow)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# (Optional) if you still see run-to-run differences on GPU, uncomment to force CPU:
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Reduce non-determinism from multi-threaded math libraries
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

random.seed(SEED)
np.random.seed(SEED)

print("Determinism bootstrap set. SEED =", SEED)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import fetch_olivetti_faces
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score

# Reproducibility (TensorFlow + NumPy)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Warning: could not enable TF op determinism:", e)

# Keep TF single-threaded for stability (optional, but helps)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except Exception as e:
    print("Warning: could not set TF threading:", e)


## 1. Load and visualize the dataset

In [ ]:
data = fetch_olivetti_faces(shuffle=False)
X = data.data.astype(np.float32)   # (400, 4096), values in [0,1]
y = data.target.astype(int)        # 40 identities, 10 images each

X.shape, y.shape


In [ ]:
def show_faces(X_flat, indices, title):
    n = len(indices)
    fig, axes = plt.subplots(1, n, figsize=(1.5*n, 2))
    fig.suptitle(title)
    for ax, i in zip(axes, indices):
        ax.imshow(X_flat[i].reshape(64, 64), cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

for i in range(40):
    # Show all 10 images of identity i
    show_faces(X, indices=list(range(i*10,(i+1)*10)), title="Identity " + str(i) + ": 10 images")


## 2. Identity Consistency Score (ICS) — provided

This is the same ICS definition as in the competitive-learning assignment.
We provide the implementation here so you can reuse it.


In [ ]:
def identity_consistency_score(z, y):
    """Identity Consistency Score (ICS)."""
    scores = []
    for identity in np.unique(y):
        idx = np.where(y == identity)[0]
        _, counts = np.unique(z[idx], return_counts=True)
        scores.append(np.max(counts) / len(idx))
    return float(np.mean(scores))


## 3. Competitive learning helper (do not modify)

In [ ]:
def fit_online_kmeans(X_feat, n_clusters, n_epochs=6, random_state=0):
    n_clusters = int(n_clusters)
    model = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=1,
        max_iter=1,
        init="random",
        random_state=int(random_state),
        n_init="auto",
    )

    # Initialization batch (required by scikit-learn)
    rng0 = np.random.RandomState(int(random_state))
    init_idx = rng0.choice(len(X_feat), size=n_clusters, replace=False)
    model.partial_fit(X_feat[init_idx])

    # Online updates
    for epoch in range(int(n_epochs)):
        rng = np.random.RandomState(int(random_state) + epoch)
        for i in rng.permutation(len(X_feat)):
            model.partial_fit(X_feat[i:i+1])

    return model


## 4. Baseline (provided): ICS from competitive learning on raw pixels

In [ ]:
k = 9
model_raw = fit_online_kmeans(X, n_clusters=k, n_epochs=6, random_state=0)
z_raw = model_raw.predict(X)
ics_raw = identity_consistency_score(z_raw, y)

ics_raw


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert len(z_raw) == 400, "z_raw must have length 400"
assert 0.0 <= ics_raw <= 1.0, "ics_raw must be in [0,1]"
print("Sanity check passed: raw baseline computed (ics_raw =", round(ics_raw, 4), ")")


## 5. Train an autoencoder (TensorFlow/Keras)

Use:
- `latent_dim = 32`
- Only one hidden layer both in the encoder, and in the decoder, hidden size = 256
- Adam(learning_rate=1e-3)
- MSE loss
- epochs = 15
- batch size = 64
- No activation on the latent layer
- ReLU activation on the hidden layers
- Q: Which activation on the output layer?


In [ ]:
# Write here your own code.
# Requirements:
# - define: latent_dim = 32
# - build: encoder and autoencoder (Keras models)
# - compile with Adam(learning_rate=1e-3) and MSE loss
# - train for 15 epochs with batch_size=64 and shuffle=False (no labels)
# - store final epoch training loss (float) in: final_train_loss
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'final_train_loss' in globals(), "final_train_loss not defined"
assert np.isfinite(final_train_loss) and final_train_loss > 0, "final_train_loss must be finite and > 0"
print("Sanity check passed: final_train_loss is valid:", final_train_loss)


### **Question 1**
Report the **final training loss** (MSE) after 15 epochs (rounded to 2 decimals).


In [ ]:
# ANSWER Q1
Q1_final_train_loss = round(float(final_train_loss), 2)
Q1_final_train_loss


## 5b. Visualize reconstructions (not graded)

In [ ]:
import matplotlib.pyplot as plt

# Write here your own code.
# Requirements:
# - Use fixed indices, e.g.: idx = [0, 1, 3, 4, 30, 32, 33, 35, 60, 61, 63, 65, 70, 72, 73, 79]
# - Compute reconstructions using your trained autoencoder
# - Plot originals and reconstructions in a 2 x 8 grid
#
# Hint:
#   X_hat = autoencoder.predict(X[idx], verbose=0)
#
# YOUR CODE HERE


## 6. Latent representations and ICS in latent space

**Task:** Compute `Z_latent` (shape `(400, 32)`), then run online k-means on `Z_latent` (k=9, 6 epochs, seed 0)
and compute `ics_lat`.


In [ ]:
# Write here your own code.
# Requirements:
# - define: Z_latent as numpy array of shape (400, latent_dim)
# - fit model_lat = fit_online_kmeans(Z_latent, n_clusters=9, n_epochs=6, random_state=0)
# - define z_lat = model_lat.predict(Z_latent)
# - compute ics_lat = identity_consistency_score(z_lat, y)
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'latent_dim' in globals() and latent_dim == 32, "latent_dim must be 32"
assert 'Z_latent' in globals(), "Z_latent not defined"
assert isinstance(Z_latent, np.ndarray), "Z_latent must be a numpy array"
assert Z_latent.shape == (400, 32), "Z_latent must have shape (400, 32)"
assert 'ics_lat' in globals(), "ics_lat not defined"
assert 0.0 <= ics_lat <= 1.0, "ics_lat must be in [0,1]"
print("Sanity check passed: Z_latent and ics_lat look valid.")


### **Question 2**
What is the **ICS** in latent space (rounded to 2 decimals)?


In [ ]:
# ANSWER Q2
Q2_ics_latent = round(float(ics_lat), 2)
Q2_ics_latent


### **Question 3**
Compute \(\Delta = \mathrm{ICS}_{latent} - \mathrm{ICS}_{raw}\) (rounded to 2 decimals).


In [ ]:
# ANSWER Q3
Q3_delta_ics = round(float(ics_lat - ics_raw), 2)
Q3_delta_ics


## 7. Stability in latent space (ARI across seeds)

Train another latent-space k-means with `random_state = 1` (same k and epochs), then compute:
- `ari_lat = adjusted_rand_score(z_lat_seed0, z_lat_seed1)`.


In [ ]:
# Write here your own code.
# Requirements:
# - fit model_lat_seed1 on Z_latent with random_state=1
# - define z_lat_seed0, z_lat_seed1
# - compute ari_lat
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'ari_lat' in globals(), "ari_lat not defined"
assert -1.0 <= ari_lat <= 1.0, "ARI must be between -1 and 1"
print("Sanity check passed: ari_lat in valid range:", ari_lat)


### **Question 4**
What is the **ARI** between latent-space clusterings (seed 0 vs seed 1), rounded to 2 decimals?


In [ ]:
# ANSWER Q4
Q4_ari_latent_seed0_vs_seed1 = round(float(ari_lat), 2)
Q4_ari_latent_seed0_vs_seed1


## Submission checklist

Your notebook must define:

- `Q1_final_train_loss`
- `Q2_ics_latent`
- `Q3_delta_ics`
- `Q4_ari_latent_seed0_vs_seed1`
